# Complete Guide: Installing OpenMC with Conda

> **OpenMC** is an open-source Monte Carlo particle transport code used for nuclear reactor simulations and neutron/photon transport problems.

This notebook walks you through the **full installation process** using `conda`, from environment setup to verification — including troubleshooting tips for common issues.

---

## Table of Contents
1. [Prerequisites](#prerequisites)
2. [Step 1 – Install Conda (Miniconda)](#step1)
3. [Step 2 – Create a Dedicated Environment](#step2)
4. [Step 3 – Add the conda-forge Channel](#step3)
5. [Step 4 – Install OpenMC](#step4)
6. [Step 5 – Install Nuclear Data Libraries](#step5)
7. [Step 6 – Configure the Data Path](#step6)
8. [Step 7 – Verify the Installation](#step7)
9. [Step 8 – Jupyter Kernel Setup (Optional)](#step8)
10. [🛠 Troubleshooting](#troubleshooting)

---
## 1. Prerequisites <a id='prerequisites'></a>

Before you begin, make sure you have:

| Requirement | Details |
|---|---|
| **OS** | Linux (recommended), macOS, or Windows (via WSL2) |
| **RAM** | ≥ 4 GB (8 GB+ recommended for large simulations) |
| **Disk space** | ≥ 5 GB free (nuclear data libraries are large) |
| **Conda** | Miniconda or Anaconda installed |
| **Internet** | Required to download packages and data |

### ⚠️ Windows Users
OpenMC is **not natively supported on Windows**. Use one of these approaches:
- **WSL2** (Windows Subsystem for Linux) — *strongly recommended*
- Docker container

To enable WSL2 on Windows, open PowerShell as Administrator and run:
```powershell
wsl --install
```
Then restart your machine and continue inside a Linux terminal.

---
## Step 1 – Install Conda (Miniconda) <a id='step1'></a>

If you already have `conda` installed, skip to [Step 2](#step2).

### Linux / WSL2
```bash
# Download the Miniconda installer
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh

# Make it executable and run it
bash Miniconda3-latest-Linux-x86_64.sh

# Follow the prompts, then reload your shell
source ~/.bashrc
```

### macOS (Apple Silicon / M1/M2/M3)
```bash
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-arm64.sh
bash Miniconda3-latest-MacOSX-arm64.sh
source ~/.zshrc
```

### macOS (Intel)
```bash
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-x86_64.sh
bash Miniconda3-latest-MacOSX-x86_64.sh
source ~/.zshrc
```

### Verify conda is working
```bash
conda --version
# Expected output: conda 23.x.x  (or similar)
```

---
## Step 2 – Create a Dedicated Conda Environment <a id='step2'></a>

Always install OpenMC in its **own isolated environment** to avoid dependency conflicts with other projects.

```bash
# Create a new environment named 'openmc-env' with Python 3.10
conda create -n openmc-env python=3.10
```

When prompted `Proceed ([y]/n)?`, type `y` and press Enter.

Then activate the environment:
```bash
conda activate openmc-env
```

Your terminal prompt should now show `(openmc-env)` at the beginning.

> 💡 **Why Python 3.10?** OpenMC's Python API has been most extensively tested on Python 3.9–3.11. Avoid Python 3.12+ until official support is confirmed.

---
## Step 3 – Add the conda-forge Channel <a id='step3'></a>

OpenMC is distributed through the **conda-forge** community channel, not the default Anaconda channel.

```bash
# Add conda-forge and set it as the highest priority channel
conda config --add channels conda-forge
conda config --set channel_priority strict
```

Verify your channel configuration:
```bash
conda config --show channels
# Expected output:
# channels:
#   - conda-forge
#   - defaults
```

> 💡 Setting `channel_priority strict` ensures conda-forge packages are preferred over `defaults`, which prevents mixing incompatible builds.

---
## Step 4 – Install OpenMC <a id='step4'></a>

Make sure your `openmc-env` environment is active, then run:

```bash
conda install -c conda-forge openmc
```

This will install OpenMC along with all required dependencies, including:
- HDF5 libraries
- NumPy, h5py, lxml, matplotlib
- OpenMC's C++ compiled core

Installation may take **5–15 minutes** depending on your connection speed.

### Installing specific versions
If you need a specific version of OpenMC (e.g., for reproducibility):
```bash
# List available versions
conda search -c conda-forge openmc

# Install a specific version
conda install -c conda-forge openmc=0.14.0
```

### Verify the installation
```bash
openmc --version
# Expected: OpenMC version X.X.X
```

---
## Step 5 – Install Nuclear Data Libraries <a id='step5'></a>

OpenMC **requires nuclear data** to run any simulation. The most common library is **ENDF/B-VIII.0** from NNDC. There are two ways to get it.

---

### Option A: Download via OpenMC's built-in script (Recommended)

```bash
# Create a directory to store nuclear data
mkdir -p ~/nuclear_data
cd ~/nuclear_data

# Use OpenMC's helper script to download ENDF/B-VIII.0 (HDF5 format)
python -c "import openmc.data; openmc.data.download_endf()"
```

> ⚠️ This will download ~2–4 GB of data. Make sure you have sufficient disk space and a stable connection.

---

### Option B: Download pre-processed HDF5 libraries manually

Pre-processed HDF5 versions are available from the OpenMC data repository:

```bash
mkdir -p ~/nuclear_data
cd ~/nuclear_data

# Download ENDF/B-VIII.0 HDF5 cross sections (~2.1 GB)
wget https://anl.box.com/shared/static/uhbxlrx7hvxqw27psymfbhi7bx7s6u6a.xz \
  -O endfb-viii.0-hdf5.tar.xz

# Extract the archive
tar -xvf endfb-viii.0-hdf5.tar.xz
```

After extraction, you should have a directory like:
```
~/nuclear_data/endfb-viii.0-hdf5/
    cross_sections.xml
    neutron/
    photon/
    ...
```

---

### Available Data Libraries

| Library | Description | Recommended for |
|---|---|---|
| **ENDF/B-VIII.0** | US evaluated nuclear data | General use |
| **JEFF-3.3** | European nuclear data | European reactor designs |
| **JENDL-4.0** | Japanese nuclear data | Japanese reactor designs |
| **TENDL** | TALYS-based evaluated nuclear data | Activation calculations |

---
## Step 6 – Configure the Data Path <a id='step6'></a>

OpenMC needs to know **where to find** your nuclear data. There are two ways to configure this:

---

### Method A: Environment Variable (Recommended)

Set the `OPENMC_CROSS_SECTIONS` environment variable to point to your `cross_sections.xml` file:

```bash
# Add this line to your ~/.bashrc (Linux) or ~/.zshrc (macOS)
echo 'export OPENMC_CROSS_SECTIONS="$HOME/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml"' >> ~/.bashrc

# Reload the shell configuration
source ~/.bashrc
```

Verify the variable is set:
```bash
echo $OPENMC_CROSS_SECTIONS
# Expected: /home/youruser/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml
```

---

### Method B: Set the Path Inside Python

You can also set the path programmatically at the start of each script or notebook:

In [ ]:
import openmc
import os

# Set the cross sections path directly in Python
os.environ['OPENMC_CROSS_SECTIONS'] = os.path.expanduser(
    '~/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml'
)

print('Cross sections path:', os.environ['OPENMC_CROSS_SECTIONS'])

---
## Step 7 – Verify the Installation <a id='step7'></a>

Run the following checks to confirm everything is working correctly.

In [ ]:
# Check 1: Import OpenMC
import openmc
print('OpenMC version:', openmc.__version__)
print('✅ OpenMC imported successfully')

In [ ]:
# Check 2: Verify nuclear data is accessible
import os

xs_path = os.environ.get('OPENMC_CROSS_SECTIONS', 'NOT SET')
print('Cross sections path:', xs_path)

if xs_path != 'NOT SET' and os.path.exists(xs_path):
    print('✅ Nuclear data library found!')
else:
    print('❌ Cross sections file not found. Check your OPENMC_CROSS_SECTIONS path.')

In [ ]:
# Check 3: Run a minimal OpenMC simulation (infinite medium)
import openmc

# --- Materials ---
uo2 = openmc.Material(name='UO2 fuel at 2.4% enrichment')
uo2.set_density('g/cc', 10.29769)
uo2.add_element('U', 1.0, enrichment=2.4)
uo2.add_element('O', 2.0)
materials = openmc.Materials([uo2])
materials.export_to_xml()

# --- Geometry ---
sphere = openmc.Sphere(r=100.0, boundary_type='reflective')
cell = openmc.Cell(fill=uo2, region=-sphere)
universe = openmc.Universe(cells=[cell])
geometry = openmc.Geometry(universe)
geometry.export_to_xml()

# --- Settings ---
settings = openmc.Settings()
settings.batches = 10
settings.inactive = 5
settings.particles = 1000
settings.run_mode = 'eigenvalue'
bounds = [-100, -100, -100, 100, 100, 100]
uniform_dist = openmc.stats.Box(bounds[:3], bounds[3:])
settings.source = openmc.IndependentSource(space=uniform_dist)
settings.export_to_xml()

# --- Run ---
openmc.run()
print('\n✅ Simulation completed successfully!')

---
## Step 8 – Jupyter Kernel Setup (Optional) <a id='step8'></a>

If you want to use your `openmc-env` environment as a Jupyter kernel (so you can select it in Jupyter Lab/Notebook):

```bash
# Make sure the environment is active
conda activate openmc-env

# Install ipykernel inside the environment
conda install -c conda-forge ipykernel

# Register it as a Jupyter kernel
python -m ipykernel install --user --name openmc-env --display-name "Python (OpenMC)"
```

Now when you launch Jupyter Lab or Notebook, you can select **"Python (OpenMC)"** from the kernel menu.

```bash
# Launch Jupyter Lab
jupyter lab
```

---
## 🛠 Troubleshooting <a id='troubleshooting'></a>

---

### ❌ Error: `PackagesNotFoundError: openmc`

**Cause:** The `conda-forge` channel is not enabled or has lower priority.

**Fix:**
```bash
conda config --add channels conda-forge
conda config --set channel_priority strict
conda install -c conda-forge openmc
```

---

### ❌ Error: `FileNotFoundError: Cross sections file does not exist`

**Cause:** `OPENMC_CROSS_SECTIONS` is not set or points to the wrong path.

**Fix:**
```bash
# Check if the variable is set
echo $OPENMC_CROSS_SECTIONS

# Verify the file actually exists at that path
ls -lh $OPENMC_CROSS_SECTIONS

# If not, re-set the correct path
export OPENMC_CROSS_SECTIONS="/correct/path/to/cross_sections.xml"
```

---

### ❌ Error: `libhdf5.so: cannot open shared object file`

**Cause:** HDF5 library is missing or not linked correctly.

**Fix:**
```bash
# Reinstall HDF5 and h5py inside your conda environment
conda install -c conda-forge hdf5 h5py
```

---

### ❌ Error: `ImportError: No module named 'openmc'` in Jupyter

**Cause:** Jupyter is using a different Python kernel (not the `openmc-env` one).

**Fix:**
1. Check the top-right corner of Jupyter — it should say **"Python (OpenMC)"**
2. If not, go to **Kernel → Change Kernel → Python (OpenMC)**
3. If the kernel doesn't appear, re-run the kernel registration:
```bash
conda activate openmc-env
python -m ipykernel install --user --name openmc-env --display-name "Python (OpenMC)"
```

---

### ❌ Error: `CondaHTTPError` or `SSLError` during install

**Cause:** Network issues or SSL certificate problems.

**Fix:**
```bash
# Option 1: Try updating conda first
conda update conda

# Option 2: Disable SSL verification temporarily (not recommended for production)
conda config --set ssl_verify false
conda install -c conda-forge openmc
conda config --set ssl_verify true  # re-enable afterward
```

---

### ❌ Error: `Solving environment: failed` (dependency conflicts)

**Cause:** Package version conflicts in an existing environment.

**Fix:** Use a **fresh environment** — this is the most reliable solution:
```bash
# Remove the old environment
conda deactivate
conda remove -n openmc-env --all

# Create a new clean environment
conda create -n openmc-env python=3.10
conda activate openmc-env
conda install -c conda-forge openmc
```

---

### ❌ Error: `openmc: command not found` after installation

**Cause:** The environment is not activated.

**Fix:**
```bash
conda activate openmc-env
openmc --version
```

---

### ❌ OpenMC runs but gives wrong k-effective values

**Cause:** Using wrong or mismatched nuclear data library version.

**Fix:**
- Confirm the library version matches your simulation requirements
- Check `cross_sections.xml` to see which isotopes and temperatures are available
- Increase the number of particles and batches for better statistics:
```python
settings.particles = 10000  # increase from 1000
settings.batches = 100
settings.inactive = 20
```

---

### ⚡ Performance Tips

- OpenMC supports **multi-threading** automatically using OpenMP. To control the number of threads:
```bash
export OMP_NUM_THREADS=8   # use 8 CPU cores
openmc                     # or run from Python
```
- For MPI parallelism (cluster use), install the MPI-enabled build:
```bash
conda install -c conda-forge openmc=*=mpi*
```

---
## ✅ Quick Reference Checklist

```
[ ] Conda installed and working
[ ] New environment created: conda create -n openmc-env python=3.10
[ ] Environment activated: conda activate openmc-env
[ ] conda-forge channel added with strict priority
[ ] OpenMC installed: conda install -c conda-forge openmc
[ ] Nuclear data downloaded (ENDF/B-VIII.0 recommended)
[ ] OPENMC_CROSS_SECTIONS environment variable set in ~/.bashrc
[ ] Jupyter kernel registered (if using Jupyter)
[ ] Verification test run successfully
```

---

## 📚 Further Resources

| Resource | Link |
|---|---|
| Official OpenMC Docs | https://docs.openmc.org |
| OpenMC GitHub | https://github.com/openmc-dev/openmc |
| OpenMC User Forum | https://openmc.discourse.group |
| Nuclear Data Library | https://www.nndc.bnl.gov/endf |
| OpenMC Workshop Notebooks | https://github.com/openmc-dev/openmc-notebooks |